### Geração de Dados Fictícios 

In [0]:
# Instalando biblioteca para geração de dados fictícios
%pip install faker

In [0]:
# Importando bibliotecas necessárias
from pyspark.sql import SparkSession
import random
from faker import Faker

# Inicializando SparkSession e Faker com locale brasileiro
spark = SparkSession.builder.getOrCreate()
fake = Faker("pt_BR")

In [0]:
# Gerando dados de 40 agências
agencias = [
    (i, f"Agência {i}", fake.city(), fake.state_abbr()) 
    for i in range(1, 41)
]

# Cria DataFrame com os dados gerados
df_agencias = spark.createDataFrame(
    agencias, 
    ["agencia_id", "nome_agencia", "cidade", "uf"]
)

# Salva como tabela Delta
df_agencias.write.mode("overwrite").saveAsTable("agencias")

In [0]:
%python
# Sortear 30 clientes diferentes para cada tipo de nulo
clientes_sem_num_parcelas = set(random.sample(range(1, 501), 30))
clientes_sem_parcelas_pagas = set(random.sample(range(1, 501), 30)) - clientes_sem_num_parcelas
clientes_sem_vencimento = set(random.sample(range(1, 501), 30))

# Gerando 500 clientes únicos
clientes = []
for i in range(1, 501):
    agencia_id = random.randint(1, 40)
    
    # Número de parcelas: nulo apenas para os 30 clientes sorteados
    num_parcelas = None if i in clientes_sem_num_parcelas else random.randint(6, 60)
    
    # Parcelas pagas: nulo para 30 clientes sorteados (exceto se num_parcelas já for nulo)
    parcelas_pagas = None if (i in clientes_sem_parcelas_pagas or num_parcelas is None) else random.randint(0, num_parcelas)
    
    # Vencimento: nulo para 30 clientes sorteados
    vencimento = None if i in clientes_sem_vencimento else str(fake.date_between(start_date="today", end_date="+2m"))
    
    clientes.append((
        i,
        fake.name(),
        fake.cpf(),
        fake.email() if random.random() > 0.005 else None,  # 0.5% nulos
        fake.phone_number() if random.random() > 0.002 else None,  # 0.2% nulos
        fake.address() if random.random() > 0.02 else None,  # 2% nulos
        agencia_id,
        str(fake.date_between(start_date="-2y", end_date="-6m")),  # Data de cadastro
        num_parcelas,
        parcelas_pagas,
        vencimento
    ))

# Particionando para 12 meses (mesmos clientes em cada mês)
clientes_particionado = []
for mes in range(12):
    data_particao = (datetime.today() - timedelta(days=30 * mes)).strftime("%Y-%m-%d")
    for c in clientes:
        clientes_particionado.append(c + (data_particao,))

# Criando DataFrame
df_clientes = spark.createDataFrame(
    clientes_particionado,
    ["cliente_id", "nome", "cpf_cnpj", "email", "telefone", "endereco",
     "agencia_id", "data_cadastro", "numero_parcelas", "parcelas_pagas",
     "vencimento", "data_particao"]
)

# Salvando como tabela Delta particionada
df_clientes.write \
    .mode("overwrite") \
    .partitionBy("data_particao") \
    .saveAsTable("clientes")

In [0]:
%python
# Tipos de contratos do mercado financeiro
tipos_contrato = [
    "Seguro", 
    "Financiamento", 
    "Empréstimo Pessoal", 
    "Consignado", 
    "Cartão de Crédito"
]

# Tipos de garantia para os contratos
tipos_garantia = ["Imóvel", "Veículo", "Aplicações Financeiras"]

# Gerando 800 contratos
contratos = []
for i in range(1, 801):
    cliente_id = random.randint(1, 500)
    agencia_id = clientes[cliente_id - 1][6]  # Mesma agência do cliente (índice 6)
    
    # Gerando dados do contrato
    tipo_contrato = random.choice(tipos_contrato)
    valor_total = round(random.uniform(5000, 50000), 2)
    qtd_parcelas = random.randint(6, 60)
    valor_parcela = round(valor_total / qtd_parcelas, 2)
    parcelas_em_atraso = random.randint(0, min(5, qtd_parcelas))  # Até 5 parcelas atrasadas
    
    # Garantia vinculada
    if tipo_contrato == "Financiamento":
        garantia = random.choice(tipos_garantia) if random.random() > 0.05 else None  # 95% com garantia
    else:
        garantia = random.choice(tipos_garantia) if random.random() > 0.015 else None  # 98.5% com garantia
    
    # Datas do contrato
    data_contrato = fake.date_between(start_date="-2y", end_date="-3m")
    data_inicio = data_contrato + timedelta(days=random.randint(1, 15))
    data_fim = data_inicio + timedelta(days=qtd_parcelas * 30)  # Aproximadamente mensal
    
    contratos.append((
        i,
        cliente_id,
        agencia_id,
        tipo_contrato,
        garantia,
        str(data_contrato),
        str(data_inicio),
        str(data_fim),
        valor_total,
        valor_parcela,
        qtd_parcelas,
        parcelas_em_atraso
    ))

# Particionando para 12 meses
contratos_particionado = []
for mes in range(12):
    data_particao = (datetime.today() - timedelta(days=30 * mes)).strftime("%Y-%m-%d")
    for ct in contratos:
        contratos_particionado.append(ct + (data_particao,))

# Criando DataFrame
df_contratos = spark.createDataFrame(
    contratos_particionado,
    ["contrato_id", "cliente_id", "agencia_id", "tipo_contrato", "garantia",
     "data_contrato", "data_inicio", "data_fim", "valor_total",
     "valor_parcela", "quantidade_parcelas", "parcelas_em_atraso", "data_particao"]
)

# Salvando como tabela Delta particionada
df_contratos.write \
    .mode("overwrite") \
    .partitionBy("data_particao") \
    .saveAsTable("contratos")